# カメラ外部キャリブレーション 概念デモ（numpy）

`extrinsic_calibration.md` の理論を **numpyだけ** で手を動かして確認する。OpenCV不要。

1. 投影パイプライン `u = K[R|t]X` と、R・t を動かすと像がどう動くか
2. DLT で PnP を解く（合成データ）→ 真の R,t と比較
3. エピポーラ幾何（F行列とエピポーラ線）
4. ステレオ・レクティフィケーション → 左右の縦座標 v が一致することを検証
5. 視差 → 深度 `Z=fB/d` の確認

> 実APIは `extrinsic_calibration_opencv.ipynb` で。

In [ ]:
# %pip install numpy matplotlib
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)

def rotx(a): c,s=np.cos(a),np.sin(a); return np.array([[1,0,0],[0,c,-s],[0,s,c]])
def roty(a): c,s=np.cos(a),np.sin(a); return np.array([[c,0,s],[0,1,0],[-s,0,c]])
def rotz(a): c,s=np.cos(a),np.sin(a); return np.array([[c,-s,0],[s,c,0],[0,0,1]])
print('ready')

---

## 1. 投影パイプライン `u = K[R|t]X`

内部行列 K と外部 [R|t] を定義し、3D点を画像へ投影する。

In [ ]:
# 内部パラメータ
fx=fy=800.0; cx,cy=640.0,360.0
K = np.array([[fx,0,cx],[0,fy,cy],[0,0,1]])

# 外部パラメータ: world->camera。カメラをワールド点群の前方に置く
R = rotx(np.deg2rad(-160))            # 下を向ける向き（例）
C = np.array([0.0, -3.0, 0.0])        # カメラ中心（ワールド座標）
t = -R @ C                            # 並進 t = -R C （§2.1）
print('t =', t, '  →  復元した中心 C=-R^T t =', -R.T@t)

def project(K, R, t, Xw):
    Xw = np.atleast_2d(Xw)
    Xc = (R @ Xw.T + t[:,None]).T      # world->camera
    x = Xc[:, :2] / Xc[:, 2:3]         # 透視投影（Zで割る）
    u = (K @ np.c_[x, np.ones(len(x))].T).T
    return u[:, :2], Xc[:,2]            # ピクセル, 深度

# 地面の格子点（z=0 平面に並べる）
gx, gy = np.meshgrid(np.linspace(-2,2,9), np.linspace(0,6,9))
Xw = np.c_[gx.ravel(), gy.ravel(), np.zeros(gx.size)]
uv, depth = project(K, R, t, Xw)
print('投影点数:', len(uv), ' 深度範囲:', depth.min().round(2), depth.max().round(2))

In [ ]:
plt.figure(figsize=(7,4))
plt.scatter(uv[:,0], uv[:,1], c=depth, cmap='viridis')
plt.gca().invert_yaxis(); plt.colorbar(label='depth Zc [m]')
plt.xlim(0,1280); plt.ylim(720,0)
plt.title('地面格子の投影像（色=深度）'); plt.xlabel('u [px]'); plt.ylabel('v [px]'); plt.show()

In [ ]:
# R, t を動かすと像がどう動くか：カメラを横にずらす
fig, axes = plt.subplots(1,3, figsize=(13,3.2))
for ax, dx in zip(axes, [-1.5, 0.0, 1.5]):
    C2 = np.array([dx, -3.0, 0.0]); t2 = -R @ C2
    uv2,_ = project(K, R, t2, Xw)
    ax.scatter(uv2[:,0], uv2[:,1], s=8)
    ax.set_xlim(0,1280); ax.set_ylim(720,0); ax.set_title(f'カメラ中心 x={dx}m')
plt.suptitle('外部パラメータ(並進)を変えると像が平行移動'); plt.tight_layout(); plt.show()

---

## 2. DLT で PnP を解く

K 既知のもとで、3D-2D対応から `[R|t]` を復元する（§4.1）。真値と比較して正しさを確認。

In [ ]:
def dlt_pnp(K, Xw, uv):
    # 正規化座標へ（K^-1 を掛ける）
    Ki = np.linalg.inv(K)
    xn = (Ki @ np.c_[uv, np.ones(len(uv))].T).T[:, :2]
    # s*[xn,1] = [R|t] [Xw,1] から 2式/点 を立てて A p = 0
    A=[]
    for (X,Y,Z),(x,y) in zip(Xw, xn):
        A.append([X,Y,Z,1,0,0,0,0, -x*X,-x*Y,-x*Z,-x])
        A.append([0,0,0,0,X,Y,Z,1, -y*X,-y*Y,-y*Z,-y])
    A=np.array(A)
    _,_,Vt=np.linalg.svd(A)
    M=Vt[-1].reshape(3,4)               # [R|t] (スケール不定)
    R_=M[:,:3]; t_=M[:,3]
    # R を直交化（SVD）し、スケールを回復
    U,S,Vt2=np.linalg.svd(R_)
    R_ortho=U@Vt2
    scale=1.0/np.mean(S)
    t_=t_*scale
    if np.linalg.det(R_ortho)<0: R_ortho=-R_ortho; t_=-t_
    # 点が前方(Z>0)になるよう符号調整
    if (R_ortho@Xw[0]+t_)[2] < 0: R_ortho=-R_ortho; t_=-t_
    return R_ortho, t_

# 一般姿勢の真値で検証（非平面の3D点を使う）
Xw3 = np.random.uniform(-2,2,(30,3)) + np.array([0,0,8.0])
R_gt = rotz(0.2)@roty(-0.15)@rotx(0.1)
t_gt = np.array([0.5,-0.3,1.0])
uv_gt = (K@( (R_gt@Xw3.T+t_gt[:,None]) ))
uv_gt = (uv_gt[:2]/uv_gt[2]).T

R_est, t_est = dlt_pnp(K, Xw3, uv_gt)
print('R 誤差(deg):', np.rad2deg(np.arccos((np.trace(R_gt.T@R_est)-1)/2)).round(4))
print('t 真値:', t_gt, ' / 推定:', t_est.round(4))

In [ ]:
# 再投影誤差で評価
rep = (K@(R_est@Xw3.T+t_est[:,None])); rep=(rep[:2]/rep[2]).T
err = np.linalg.norm(rep-uv_gt, axis=1)
print(f'再投影誤差 平均 {err.mean():.4e} px / 最大 {err.max():.4e} px（ノイズ無し→ほぼ0）')

---

## 3. エピポーラ幾何

2視点の相対 `R,t` から E,F を作り、エピポーラ線を描く（§5）。

In [ ]:
def skew(v): return np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])

# cam1 = 基準, cam2 = 相対回転R12・並進t12
R12 = roty(np.deg2rad(-10))
t12 = np.array([1.0, 0.0, 0.0])         # 主に横方向の基線
E = skew(t12) @ R12                       # エッセンシャル行列
F = np.linalg.inv(K).T @ E @ np.linalg.inv(K)   # ファンダメンタル行列

# 3D点を両カメラに投影
P = np.random.uniform(-2,2,(12,3))+np.array([0,0,8.0])
u1=(K@P.T); u1=(u1[:2]/u1[2]).T
P2=(R12@P.T+t12[:,None]); u2=(K@P2); u2=(u2[:2]/u2[2]).T

# 拘束 u2^T F u1 = 0 を確認
cons=[np.r_[u2[i],1]@F@np.r_[u1[i],1] for i in range(len(P))]
print('エピポーラ拘束 |u2^T F u1| 最大:', np.max(np.abs(cons)))

In [ ]:
# cam2画像上のエピポーラ線 l2 = F u1 を描画
plt.figure(figsize=(7,4))
xs=np.array([0,1280])
for i in range(len(P)):
    l=F@np.r_[u1[i],1]            # a,b,c : a u + b v + c =0
    ys=-(l[0]*xs+l[2])/l[1]
    plt.plot(xs,ys,alpha=0.5)
plt.scatter(u2[:,0],u2[:,1],c='k',zorder=3,label='cam2の対応点')
plt.xlim(0,1280); plt.ylim(720,0); plt.legend()
plt.title('cam2上のエピポーラ線（各点はその線上に乗る）'); plt.show()

---

## 4. ステレオ・レクティフィケーション

両カメラを共通の `R_rect` に回し、**対応点の縦座標 v が左右で一致**することを検証する（§6.2）。

In [ ]:
# 2カメラをワールドに配置（中心だけ基線方向にずらし、向きは少し異なる）
C_l = np.array([0.0,0.0,0.0]); C_r = np.array([0.6,0.0,0.0])   # 基線0.6m
R_l = roty(np.deg2rad( 4)); R_r = roty(np.deg2rad(-4))         # 軽い輻輳
t_l = -R_l@C_l; t_r = -R_r@C_r

# 共通の rectify 回転 R_rect（世界座標, §6.2）
new_x = (C_r - C_l)/np.linalg.norm(C_r-C_l)        # 基線方向
opt_l = R_l.T@np.array([0,0,1.0]); opt_r = R_r.T@np.array([0,0,1.0])
old_z = (opt_l+opt_r); old_z/=np.linalg.norm(old_z)
new_y = np.cross(old_z,new_x); new_y/=np.linalg.norm(new_y)
new_z = np.cross(new_x,new_y)
R_rect = np.vstack([new_x,new_y,new_z])            # world->rectified

Knew = K.copy()
def project_rect(C):
    t_=-R_rect@C
    Xc=(R_rect@scene.T+t_[:,None])
    uvw=Knew@Xc; return (uvw[:2]/uvw[2]).T

scene = np.random.uniform(-1.5,1.5,(15,3))+np.array([0,0,7.0])
uvL = project_rect(C_l); uvR = project_rect(C_r)
dv = np.abs(uvL[:,1]-uvR[:,1])
print('レクティファイ後の縦ズレ |v_L - v_R| 最大:', dv.max().round(6), 'px → ≈0 なら水平整列OK')

In [ ]:
fig,(a1,a2)=plt.subplots(1,2,figsize=(12,4),sharey=True)
a1.scatter(uvL[:,0],uvL[:,1],c='b'); a1.set_title('左(rectified)')
a2.scatter(uvR[:,0],uvR[:,1],c='r'); a2.set_title('右(rectified)')
for i in range(len(scene)):                 # 同じ高さに水平線
    a1.axhline(uvL[i,1],color='gray',lw=0.3); a2.axhline(uvR[i,1],color='gray',lw=0.3)
for ax in (a1,a2): ax.set_xlim(0,1280); ax.set_ylim(720,0); ax.set_xlabel('u')
a1.set_ylabel('v'); plt.suptitle('対応点が同じ行に乗る＝1次元探索でマッチ可能'); plt.show()

---

## 5. 視差 → 深度 `Z = fB/d`

レクティファイ後の横座標差（視差）から深度を復元し、真の深度と一致するか確認（§6.3）。

In [ ]:
B = np.linalg.norm(C_r-C_l)                # 基線長
f = Knew[0,0]
disparity = uvL[:,0]-uvR[:,0]              # 視差
Z_est = f*B/disparity
# 真の深度（rectified座標でのZ）
Z_true = (R_rect@(scene.T) - R_rect@C_l[:,None])[2]
for i in range(5):
    print(f'd={disparity[i]:7.2f}px  Z_est={Z_est[i]:6.3f}m  Z_true={Z_true[i]:6.3f}m')
print('深度誤差 最大:', np.abs(Z_est-Z_true).max().round(6),'m')

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(disparity, Z_est, label='Z=fB/d')
plt.xlabel('視差 d [px]'); plt.ylabel('深度 Z [m]')
plt.title('視差が小さいほど遠い（反比例）'); plt.grid(alpha=0.3); plt.legend(); plt.show()

---

## まとめ

- `t` は並進、位置は `C=-Rᵀt`（符号に注意）
- DLT で K既知から `[R|t]` を復元できた（再投影誤差≈0）
- エピポーラ拘束 `u₂ᵀFu₁=0` を数値確認、エピポーラ線を可視化
- レクティファイで左右の `v` が一致 → 1次元マッチングが可能に
- `Z=fB/d` で視差から深度を正しく復元

次は `extrinsic_calibration_opencv.ipynb` で実APIを叩く。